# Multilingual NER Tokenization

This notebook handles the tokenization of the harmonized datasets and the alignment of BIO labels to sub-tokens.

## Environment Setup

Loading necessary libraries and configuring the compute device (CUDA/CPU).

In [1]:
import torch
import evaluate
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Tokenizer Initialization

Initializing the `xlm-roberta-base` tokenizer.
This model is chosen for its **comprehensive multilingual support** and **massive shared vocabulary**. These features are critical for the project as they enable better cross-lingual transfer between Hungarian, English, and German, even when training on a combined dataset.

In [3]:
model_id = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

## Loading Preprocessed Datasets

Loading the Interleaved, Hungarian, English, German, and  Gold_only datasets from disk.

In [5]:
interleaved_dataset = load_from_disk("../data/processed/harmonized_dataset/interleaved_dataset")
gold_only_dataset = load_from_disk("../data/processed/harmonized_dataset/gold_only_dataset")
hun_dataset = load_from_disk("../data/processed/harmonized_dataset/hun_dataset")
eng_dataset = load_from_disk("../data/processed/harmonized_dataset/eng_dataset")
ger_dataset = load_from_disk("../data/processed/harmonized_dataset/ger_dataset")

interleaved_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 75120
    })
    validation: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 5684
    })
    test: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 5684
    })
})

### Tokenization Sample

Inspecting individual examples to understand tokenization length increases.

In [6]:
example = tokenizer(interleaved_dataset['train'][0]['tokens'], is_split_into_words=True)

In [10]:
for k,v in example.items():
    print(k,"\n", v)

input_ids 
 [0, 62, 19215, 85, 864, 3288, 77087, 13, 714, 27213, 6, 4, 10, 22828, 17481, 16290, 27213, 126350, 50455, 842, 81549, 10, 191668, 14, 600, 57816, 761, 15, 113185, 48501, 14115, 152, 61242, 4, 12975, 12795, 305, 4, 2588, 186910, 198, 64830, 6, 4, 18254, 209, 154782, 4, 2588, 12795, 201, 4, 963, 186910, 35926, 67, 96804, 6, 5, 2]
attention_mask 
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [11]:
print(f"Original splitted text:\n{interleaved_dataset['train'][0]['tokens']}", end='\n\n')
print(f"Original splitted text length:\n{len(interleaved_dataset['train'][0]['tokens'])}", end='\n\n')
print(f"Tokenized text:\n{example.tokens()}", end='\n\n')
print(f"Tokenized text length:\n{len(example.tokens())}", end='\n\n')
print(f"BIO tags:\n{interleaved_dataset['train'][0]['ner']}", end='\n\n')
print(f"BIO tags length:\n{len(interleaved_dataset['train'][0]['ner'])}")

Original splitted text:
['A', 'Nasdaq', 'Composite', '25', 'pontos', ',', 'a', 'DJIA', '2,5', 'pontos', 'nyereséget', 'mutatott', 'a', 'keddi', 'nyitásban', '(', 'előző', 'zárók', ':', '1923,59', 'pont', '6,25', 'százalékos', 'eséssel', ',', 'illetve', '10208,25', 'pont', '4,10', 'százalékos', 'veszteséggel', '.']

Original splitted text length:
32

Tokenized text:
['<s>', '▁A', '▁Nas', 'da', 'q', '▁Com', 'posit', 'e', '▁25', '▁pontos', '▁', ',', '▁a', '▁DJ', 'IA', '▁2,5', '▁pontos', '▁nyere', 'séget', '▁mu', 'tatott', '▁a', '▁kedd', 'i', '▁ny', 'itás', 'ban', '▁(', '▁előző', '▁zár', 'ók', '▁:', '▁1923', ',', '59', '▁pont', '▁6', ',', '25', '▁százalékos', '▁es', 'éssel', '▁', ',', '▁illetve', '▁10', '208', ',', '25', '▁pont', '▁4', ',', '10', '▁százalékos', '▁vesz', 'te', 'séggel', '▁', '.', '</s>']

Tokenized text length:
60

BIO tags:
[0, 7, 8, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

BIO tags length:
32


In [12]:
print(example.word_ids())

[None, 0, 1, 1, 1, 2, 2, 2, 3, 4, 5, 5, 6, 7, 7, 8, 9, 10, 10, 11, 11, 12, 13, 13, 14, 14, 14, 15, 16, 17, 17, 18, 19, 19, 19, 20, 21, 21, 21, 22, 23, 23, 24, 24, 25, 26, 26, 26, 26, 27, 28, 28, 28, 29, 30, 30, 30, 31, 31, None]


## Label Alignment Function

Defining the logic to correctly map the original BIO labels to the first sub-token of each word, assigning -100 to special tokens and subsequent sub-tokens.

In [13]:
def align_labels_to_tokenized_text(ds, tokenizer, token_col, ner_col):
    tokenized_inputs = tokenizer(
        ds[token_col], 
        truncation=True, 
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(ds[ner_col]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens like <s> or </s>
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # This is the FIRST sub-token of a new word
                label_ids.append(label[word_idx])
            else:
                # This is a follow-up sub-token of the same word
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


## Bulk Tokenization & Alignment

Applying the alignment logic to all datasets in batches.

In [18]:
tokenized_dataset = interleaved_dataset.map(
    align_labels_to_tokenized_text, 
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=interleaved_dataset["train"].column_names
)


In [19]:
hun_tokenized_dataset = hun_dataset.map(
    align_labels_to_tokenized_text, 
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=hun_dataset["train"].column_names
)

In [20]:
eng_tokenized_dataset = eng_dataset.map(
    align_labels_to_tokenized_text, 
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=eng_dataset["train"].column_names
)

In [21]:
ger_tokenized_dataset = ger_dataset.map(
    align_labels_to_tokenized_text, 
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=ger_dataset["train"].column_names
)

In [22]:
gold_only_tokenized_dataset = gold_only_dataset.map(
    align_labels_to_tokenized_text, 
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=gold_only_dataset["train"].column_names
)

Map:   0%|          | 0/50080 [00:00<?, ? examples/s]

Map:   0%|          | 0/4625 [00:00<?, ? examples/s]

Map:   0%|          | 0/4625 [00:00<?, ? examples/s]

## Verification

Comparing the original and aligned BIO labels to ensure counts match.

In [23]:
print(f"Original splitted text:\n{interleaved_dataset['train'][0]['tokens']}", end='\n\n')
print(f"Original splitted text length:\n{len(interleaved_dataset['train'][0]['tokens'])}", end='\n\n')
print(f"Tokenized text:\n{example.tokens()}", end='\n\n')
print(f"Tokenized text length:\n{len(example.tokens())}", end='\n\n')
print(f"BIO tags:\n{interleaved_dataset['train'][0]['ner']}", end='\n\n')
print(f"BIO tags length:\n{len(interleaved_dataset['train'][0]['ner'])}", end='\n\n')
print(f"BIO tags after tokenization:\n{tokenized_dataset['train'][0]['labels']}", end='\n\n')
print(f"BIO tags length after tokenization:\n{len(tokenized_dataset['train'][0]['labels'])}", end='\n\n')

Original splitted text:
['A', 'Nasdaq', 'Composite', '25', 'pontos', ',', 'a', 'DJIA', '2,5', 'pontos', 'nyereséget', 'mutatott', 'a', 'keddi', 'nyitásban', '(', 'előző', 'zárók', ':', '1923,59', 'pont', '6,25', 'százalékos', 'eséssel', ',', 'illetve', '10208,25', 'pont', '4,10', 'százalékos', 'veszteséggel', '.']

Original splitted text length:
32

Tokenized text:
['<s>', '▁A', '▁Nas', 'da', 'q', '▁Com', 'posit', 'e', '▁25', '▁pontos', '▁', ',', '▁a', '▁DJ', 'IA', '▁2,5', '▁pontos', '▁nyere', 'séget', '▁mu', 'tatott', '▁a', '▁kedd', 'i', '▁ny', 'itás', 'ban', '▁(', '▁előző', '▁zár', 'ók', '▁:', '▁1923', ',', '59', '▁pont', '▁6', ',', '25', '▁százalékos', '▁es', 'éssel', '▁', ',', '▁illetve', '▁10', '208', ',', '25', '▁pont', '▁4', ',', '10', '▁százalékos', '▁vesz', 'te', 'séggel', '▁', '.', '</s>']

Tokenized text length:
60

BIO tags:
[0, 7, 8, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

BIO tags length:
32

BIO tags after tokenization:
[-1

## Saving Tokenized Datasets

Exporting the final datasets to disk for use in model fine-tuning.

In [24]:
print(tokenized_dataset)
tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/tokenized_dataset")

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 75120
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5684
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5684
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/75120 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5684 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5684 [00:00<?, ? examples/s]

In [25]:
hun_tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/hun_tokenized_dataset")
eng_tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/eng_tokenized_dataset")
ger_tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/ger_tokenized_dataset")
gold_only_tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/gold_only_tokenized_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/11960 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1495 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1495 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8469 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1059 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/3130 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3130 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50080 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4625 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4625 [00:00<?, ? examples/s]